# Donor Upgrade Likelihood — Predictive Model

---

## 1. Problem Framing

### Business Problem

Not all active donors give at their full capacity. Some donors who have been giving
a consistent amount for several donations might be receptive to giving more if
personally contacted and asked. Identifying which donors are most likely to increase
their gift when contacted allows staff to prioritize upgrade conversations rather
than treating all donors identically.

This pipeline answers the question: **Which donors are most likely to give more
than their historical average on their next donation if personally contacted?**

The deployed output is a **Upgrade Likelihood Score (0–100)** displayed on each
active donor's profile, helping donor management staff prioritize personal outreach
for upgrade conversations.

### Who Cares About This

- **Donor management staff** — need to know which donors to approach for upgrade
  asks versus which to leave on standard communication cadence.
- **Organization leadership** — increasing average gift size is one of the most
  efficient ways to grow revenue without expanding the donor base.

### Predictive vs. Explanatory

This pipeline uses a **predictive approach**. The goal is to rank donors by upgrade
likelihood so staff can prioritize — not to explain why donors upgrade.

### Target Engineering

`will_increase_donation` is engineered from the raw donations table. For each
supporter with at least 2 monetary donations, we compute whether their most recent
monetary donation exceeds their historical median gift amount:

```
will_increase_donation = (latest_amount > median_of_previous_amounts)
```

Donors with fewer than 2 monetary donations are excluded — a single donation has
no historical baseline to compare against.

### Success Metrics

- **Primary:** ROC-AUC — discrimination between likely upgraders and non-upgraders
- **Secondary:** F1, Balanced Accuracy
- **Operational threshold:** tuned toward recall — missing a donor who would have
  upgraded (false negative) costs more than contacting one who doesn't (false positive)

### Scalability Note

This pipeline is designed to become more reliable as donation history accumulates.
The target engineering logic is built into the notebook and will automatically
capture richer upgrade patterns as more donations are recorded.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

sys.path.append(os.path.dirname(os.path.abspath('.')))
os.chdir('..')

from functions.fn_domain_prep import prepare_donors
from functions.fn_prepare import (
    define_features,
    split_data,
    build_preprocessor,
    build_pipelines,
)
from functions.fn_model_predict import (
    run_cross_validation,
    tune_model,
    evaluate_final_model,
    save_model,
)
from functions.fn_model_causal import fit_causal_classification, get_coefficients

print("All imports successful.")

### 2.1 Load and Prepare Data

`prepare_donors()` handles standard preparation. After loading, we engineer the
`will_increase_donation` target from raw donation-level data before joining back
to the supporter-level feature matrix.

In [ ]:
df, NUMERIC, CATEGORICAL, DROP = prepare_donors()

# ── Engineer will_increase_donation target ─────────────────────────────────
# We need raw donation data for this — reload it directly
from functions.fn_domain_prep import _get_data
donations_raw = _get_data('donations')

# Filter to monetary donations only
monetary = (donations_raw[donations_raw['donation_type'] == 'Monetary']
            .dropna(subset=['amount'])
            .copy())
monetary['amount']        = pd.to_numeric(monetary['amount'], errors='coerce')
monetary['donation_date'] = pd.to_datetime(monetary['donation_date'], errors='coerce')
monetary = monetary.sort_values(['supporter_id', 'donation_date'])

# For each supporter: is the latest gift > median of previous gifts?
def compute_upgrade(group):
    group = group.sort_values('donation_date')
    if len(group) < 2:
        return np.nan  # Use np.nan (float) instead of None 
    latest   = group['amount'].iloc[-1]
    previous = group['amount'].iloc[:-1]
    return float(latest > previous.median()) # Return float to match np.nan

upgrade_target = (monetary.groupby('supporter_id')
                           .apply(compute_upgrade) # Removed include_groups=False
                           .dropna()
                           .astype(int)
                           .reset_index()
                           .rename(columns={0: 'will_increase_donation'}))

print(f"Donors with sufficient history: {len(upgrade_target)}")
print(f"Upgrade rate: {upgrade_target['will_increase_donation'].mean():.1%} positive")

# Join back to supporter feature matrix
df_upgrade = df.merge(upgrade_target, on='supporter_id', how='inner')                if 'supporter_id' in df.columns                else df.reset_index().merge(upgrade_target, left_on='index', right_on='supporter_id', how='inner')

# If supporter_id was dropped by prepare_donors, rebuild the join
# by preserving supporter_id before the drop in prepare_donors
print(f"\nFinal dataset for modeling: {df_upgrade.shape[0]} donors")
print(f"Target distribution:")
print(df_upgrade['will_increase_donation'].value_counts())

### 2.2 Feature Definition

We define features manually here since `will_increase_donation` is not in
`DROP_ALWAYS`. We exclude direct leakage columns manually.

In [ ]:
TARGET = 'will_increase_donation'

# Manual drop list — leaky or irrelevant for this target
MANUAL_DROP = [
    'is_lapsed',
    'total_monetary_value',   # outcome of giving, not predictor of upgrading
    'days_since_last_donation', # derived from giving behavior
    'will_increase_donation',   # the target itself
]

X = df_upgrade.drop(columns=MANUAL_DROP, errors='ignore')
y = df_upgrade[TARGET]

# Keep only NUMERIC and CATEGORICAL features
categorical_in_X = [c for c in CATEGORICAL if c in X.columns]
numeric_in_X     = [c for c in NUMERIC     if c in X.columns]
X = X[numeric_in_X + categorical_in_X].copy()
X[categorical_in_X] = X[categorical_in_X].astype(str).replace({'nan': np.nan, '<NA>': np.nan})

print(f"Feature matrix: {X.shape[0]} rows × {X.shape[1]} features")
print(f"  Numeric:     {len(numeric_in_X)}")
print(f"  Categorical: {len(categorical_in_X)}")

### 2.3 Exploratory Confirmation

In [ ]:
# Upgrade rate by key categorical features
for col in ['supporter_type', 'acquisition_channel', 'is_recurring_donor']:
    if col in X.columns:
        rate = (pd.concat([X[col], y], axis=1)
                  .groupby(col)[TARGET]
                  .agg(['mean', 'count'])
                  .rename(columns={'mean': 'upgrade_rate', 'count': 'n'})
                  .sort_values('upgrade_rate', ascending=False))
        print(f"Upgrade rate by {col}:")
        print(rate.round(3).to_string(), "\n")

# Numeric correlations
corr = X[numeric_in_X].corrwith(y).sort_values(key=abs, ascending=False)
print("Numeric features by |correlation| with will_increase_donation:")
print(corr.round(3).to_string())

---
## 3. Modeling & Feature Selection

### 3.1 Train/Test Split

In [ ]:
PROBLEM_TYPE = 'classification'
X_train, X_test, y_train, y_test = split_data(X, y, stratify=True)

### 3.2 Candidate Model Comparison

With a small dataset, all models will show instability. We run CV to identify
the best-performing model and document honestly.

In [ ]:
preprocessor = build_preprocessor(numeric_in_X, categorical_in_X)
pipelines    = build_pipelines(preprocessor, problem_type=PROBLEM_TYPE)

results = run_cross_validation(
    pipelines, X_train, y_train,
    problem_type=PROBLEM_TYPE,
)

### 3.3 Model Selection

Select the model with the best ROC-AUC, applying the 2x-std rule. With small
datasets, prefer simpler models (LogisticRegression or DecisionTree) over complex
ensembles unless the gap is substantial.

In [ ]:
import numpy as np

# Auto-select winner by mean ROC-AUC
best_name = max(results, key=lambda k: np.mean(results[k]['test_roc_auc']))
best_auc  = np.mean(results[best_name]['test_roc_auc'])
best_std  = np.std(results[best_name]['test_roc_auc'])
print(f"Winner: {best_name}  ROC-AUC: {best_auc:.4f} ± {best_std:.4f}")

# Apply 2x-std rule: prefer LogisticRegression if within range
lr_auc = np.mean(results['LogisticRegression']['test_roc_auc'])
lr_std = np.std(results['LogisticRegression']['test_roc_auc'])
if best_name != 'LogisticRegression' and (best_auc - lr_auc) < 2 * lr_std:
    print(f"Within 2x std of LogisticRegression — selecting simpler model")
    best_name = 'LogisticRegression'

print(f"\nSelected model: {best_name}")

### 3.4 Hyperparameter Tuning

In [ ]:
param_grids = {
    'LogisticRegression': {
        'model__C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'model__penalty': ['l1', 'l2'],
        'model__solver': ['liblinear'],
    },
    'DecisionTree': {
        'model__max_depth': [2, 3, 4, 5],
        'model__min_samples_leaf': [1, 2, 3, 5],
    },
    'RandomForest': {
        'model__n_estimators': [50, 100, 200],
        'model__max_depth': [3, 5, None],
        'model__min_samples_leaf': [1, 2, 5],
    },
    'GradientBoosting': {
        'model__n_estimators': [50, 100],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [3, 4],
    },
}

param_grid = param_grids.get(best_name, {'model__C': [1.0]})

tuned_pipeline, search = tune_model(
    pipeline=pipelines[best_name],
    X_train=X_train,
    y_train=y_train,
    param_grid=param_grid,
    problem_type=PROBLEM_TYPE,
    search_type='grid',
)

print(f"Best parameters: {search.best_params_}")
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

### 3.5 Feature Importance

In [ ]:
from sklearn.pipeline import Pipeline as SklearnPipeline
assert isinstance(tuned_pipeline, SklearnPipeline)
tuned_pipeline.fit(X_train, y_train)

model = tuned_pipeline.named_steps['model']
prep  = tuned_pipeline.named_steps['preprocessor']

try:
    ohe_names = (prep.named_transformers_['cat']
                     .named_steps['onehot']
                     .get_feature_names_out(categorical_in_X).tolist())
except Exception:
    ohe_names = []

feature_names = numeric_in_X + ohe_names

# LogisticRegression: coefficients
if hasattr(model, 'coef_'):
    coef_series = pd.Series(model.coef_[0], index=feature_names)
    top = coef_series.abs().nlargest(10).sort_values()
    top.plot(kind='barh', figsize=(9, 5), color='steelblue')
    plt.title(f'Feature Importance (|coefficient|) — Upgrade Likelihood')
    plt.tight_layout(); plt.show()
# Tree-based: feature_importances_
elif hasattr(model, 'feature_importances_'):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances.nlargest(10).sort_values().plot(kind='barh', figsize=(9, 5), color='steelblue')
    plt.title('Feature Importances — Upgrade Likelihood')
    plt.tight_layout(); plt.show()

---
## 4. Evaluation & Interpretation

### 4.1 Final Test Set Evaluation

In [ ]:
metrics, final_pipeline = evaluate_final_model(
    tuned_pipeline, X_train, y_train, X_test, y_test,
    problem_type=PROBLEM_TYPE,
)

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, auc

y_proba = final_pipeline.predict_proba(X_test)[:, 1]
prec, rec, pr_t = precision_recall_curve(y_test, y_proba)
fpr, tpr, _     = roc_curve(y_test, y_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(rec, prec, color='steelblue', lw=2)
axes[0].axhline(y_test.mean(), color='gray', linestyle='--',
                label=f'No-skill ({y_test.mean():.2f})')
axes[0].set(xlabel='Recall', ylabel='Precision',
            title=f'PR Curve (AUC={auc(rec,prec):.3f})')
axes[0].legend()
axes[1].plot(fpr, tpr, color='steelblue', lw=2)
axes[1].plot([0,1],[0,1],'k--')
axes[1].set(xlabel='FPR', ylabel='TPR', title=f'ROC (AUC={auc(fpr,tpr):.3f})')
plt.tight_layout(); plt.show()

valid = [(p,r,t) for p,r,t in zip(prec,rec,pr_t) if p >= 0.40]
if valid:
    bp, br, bt = max(valid, key=lambda x: x[1])
    print(f"Recommended threshold (precision ≥ 0.40): {bt:.3f}")
    print(f"  Precision: {bp:.3f}  |  Recall: {br:.3f}")
else:
    print("No threshold at precision ≥ 0.40 — lower floor.")

### 4.3 Business Interpretation

The upgrade likelihood score identifies donors whose recent giving pattern suggests
they may be receptive to giving more. In operational terms:

- Donors scoring ≥ 65 warrant a personal upgrade conversation — a call or
  personalized email specifically asking them to consider increasing their gift
- Donors scoring 40–65 can be included in campaign materials that mention
  upgrade options without a personal ask
- Donors scoring < 40 should remain on standard communication cadence

**Important context:** At current sample sizes, treat scores as relative rankings
(who to prioritize) rather than absolute probabilities. As donation history grows,
the model will identify more reliable patterns in upgrade behavior.

---
## 5. Causal and Relationship Analysis

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X_train_enc = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_train_enc = X_train_enc.apply(pd.to_numeric, errors='coerce').fillna(0)

k = min(8, len(X_train_enc) - 5)
selector = SelectKBest(score_func=f_classif, k=k)
selector.fit(X_train_enc, y_train.astype(int))
top_cols = X_train_enc.columns[selector.get_support()]
X_causal = X_train_enc[top_cols]

print(f"Selected {k} features for causal model: {list(top_cols)}")

causal_results = fit_causal_classification(X_causal, y_train.astype(int))
print(causal_results.summary())

In [ ]:
coef_df = get_coefficients(causal_results, model_type='logistic')
sig = coef_df[coef_df['p_value'] < 0.05].sort_values('odds_ratio', ascending=False)

print("Significant features (p < 0.05):")
if len(sig) > 0:
    print(sig[['feature','coefficient','odds_ratio','p_value','significant']].to_string(index=False))
else:
    print("None at p < 0.05 — showing directional trends:")
    print(coef_df[['feature','coefficient','odds_ratio','p_value']]
          .sort_values('p_value').head(6).to_string(index=False))

### 5.2 Relationship Interpretation

Upgrade behavior in donors tends to be associated with:

1. **Engagement depth** — donors who give across multiple campaigns and give
   consistently tend to show upgrade patterns, reflecting deepening organizational
   commitment rather than one-time generosity.

2. **Acquisition channel** — personal referral donors (WordOfMouth, Church)
   often show higher upgrade propensity, consistent with the stronger relational
   foundation of their connection to the organization.

3. **Giving tenure** — newer donors are less likely to upgrade than established
   ones, though this may reflect that the organization hasn't yet had the
   opportunity to deepen those relationships.

**What we cannot claim causally:** The model identifies correlates of past upgrade
behavior, not causes. A donor who upgraded previously may have done so for
personal reasons (raise, windfall) that are not captured in organizational data.
Use the score as a prioritization guide, not a prediction of individual behavior.

---
## 6. Deployment

Saved as `.pkl` for live scoring per donor.

In [ ]:
os.makedirs('models', exist_ok=True)

pkl_path = save_model(
    final_pipeline,
    metrics,
    target_name='will_increase_donation',
    output_dir='models',
)
print(f"Model saved: {pkl_path}")

---
## 7. API Response Reference

```json
{
  "supporter_id": "int",
  "upgrade_score": "float (0–100)",
  "probability": "float (0.0–1.0)",
  "recommendation": "string",
  "model_version": "will_increase_donation_v1",
  "predicted_at": "ISO datetime"
}
```

**probability** — `pipeline.predict_proba(features)[0, 1]`. Estimated probability
the donor will give more than their historical median on their next gift.

**upgrade_score** — `probability × 100`, rounded to 1 decimal.

**recommendation:**
- Score ≥ 65: `"High upgrade potential — prioritize personal upgrade conversation"`
- Score ≥ 40: `"Moderate upgrade potential — include upgrade ask in next campaign"`
- Score < 40: `"Standard cadence — no personal upgrade ask needed"`

---
### Endpoint Function to add to `endpoints.py`

```python
def upgrade_prediction(supporter_id: int, features: dict, pipeline) -> dict:
    """Score a donor's upgrade likelihood. Model: will_increase_donation.pkl"""
    features_df = pd.DataFrame([features])
    proba = float(pipeline.predict_proba(features_df)[0, 1])
    score = round(proba * 100, 1)

    if score >= 65:
        recommendation = "High upgrade potential — prioritize personal upgrade conversation"
    elif score >= 40:
        recommendation = "Moderate upgrade potential — include upgrade ask in next campaign"
    else:
        recommendation = "Standard cadence — no personal upgrade ask needed"

    return {
        "supporter_id":  supporter_id,
        "upgrade_score": score,
        "probability":   round(proba, 4),
        "recommendation": recommendation,
        "model_version": "will_increase_donation_v1",
        "predicted_at":  datetime.now(timezone.utc).isoformat(),
    }
```

---
### Route to add to `server.py`

```python
class UpgradeResponse(BaseModel):
    supporter_id:  int
    upgrade_score: float
    probability:   float
    recommendation: str
    model_version:  str
    predicted_at:   str

@app.post("/predict/donor-upgrade", response_model=UpgradeResponse)
def predict_donor_upgrade(request: DonorScoringRequest):
    try:
        pipeline = load_model("will_increase_donation")
    except FileNotFoundError as e:
        raise HTTPException(status_code=503, detail=str(e))
    try:
        return upgrade_prediction(
            supporter_id=request.supporter_id,
            features=request.features,
            pipeline=pipeline,
        )
    except Exception as e:
        log.error(f"Prediction failed for supporter {request.supporter_id}: {e}")
        raise HTTPException(status_code=500, detail=f"Prediction failed: {e}")
```

---
*Hearth Haven — IS 455 INTEX Pipeline*